# SSH remote into this Kaggle notebook

**Kaggle Secrets** (Add-ons → Secrets) — attach them to this notebook:

| Secret | Required | Content |
|---|---|---|
| `SSH_PUBLIC_KEY` | yes | Your OpenSSH public key (`ssh-ed25519 AAAA...`), several keys can go on separate lines |
| `SSH_PASSWORD` | no | Root password; when left empty, password login is disabled (key only) |
| `NGROK_TOKEN` | no | Only needed to open code-server over ngrok (cell 4) |
| `VSCODE_PASSWORD` | no | code-server password (a random one is generated when empty) |

Old names `ID_RSA_PUB`, `SSH_PASS`, `NGROK_TOKEN_1` still work. Environment variables with the same names override the secrets (handy for testing).

**Settings**: Internet **On**; pick a GPU accelerator if needed. Run cells 1 → 3, copy the printed `trycloudflare.com` link, then on Windows:
```
.\client\kaggle_ssh.ps1 https://xxxx.trycloudflare.com     # first time: add -Setup
ssh kaggle
```

In [ ]:
# 1. Fetch the code
REPO = "https://github.com/matbinhtinh/SSH_Remote_Kaggle.git"
DIR = "/kaggle/working/SSH_Remote_Kaggle"
import os, sys, subprocess
if os.path.isdir(DIR):
    subprocess.run(["git", "-C", DIR, "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO, DIR], check=True)
sys.path.insert(0, DIR)
import importlib, kaggle_setup
importlib.reload(kaggle_setup)
from kaggle_setup import *

In [ ]:
# 2. Secrets (Kaggle Secrets or environment variables)
SSH_PUBLIC_KEY  = get_secret("SSH_PUBLIC_KEY", "ID_RSA_PUB")
SSH_PASSWORD    = get_secret("SSH_PASSWORD", "SSH_PASS")
NGROK_TOKEN     = get_secret("NGROK_TOKEN", "NGROK_TOKEN_1")
VSCODE_PASSWORD = get_secret("VSCODE_PASSWORD")
print("SSH_PUBLIC_KEY :", "OK" if SSH_PUBLIC_KEY else "MISSING")
print("SSH_PASSWORD   :", "set" if SSH_PASSWORD else "(empty -> key-only login)")
print("NGROK_TOKEN    :", "set" if NGROK_TOKEN else "(empty -> code-server not published)")

In [ ]:
# 3. SSH: sshd on 127.0.0.1:2201 + Cloudflare tunnel (free, no account needed)
export_kaggle_env()                      # CUDA/PATH for SSH sessions
start_ssh(public_keys=SSH_PUBLIC_KEY, password=SSH_PASSWORD)
SSH_URL = start_cloudflared()
print_client_help(SSH_URL)

In [ ]:
# 4. (optional) code-server over ngrok HTTP
if NGROK_TOKEN:
    pw = start_vscode(password=VSCODE_PASSWORD)
    info = start_ngrok([NGROK_TOKEN])
    print("code-server:", info.get("vscode"), "| password:", pw)
else:
    print("Skipped: NGROK_TOKEN is not set")

In [ ]:
# 5. (optional) keep the session alive and watch the tunnel; stop with Interrupt
import time, datetime
while True:
    alive = subprocess.run("pgrep -f '[c]loudflared tunnel'", shell=True, capture_output=True).returncode == 0
    print(datetime.datetime.now().strftime("%H:%M:%S"), "tunnel", "OK" if alive else "DOWN -> rerun cell 3", SSH_URL, flush=True)
    time.sleep(600)